In [0]:
from pyspark.sql import functions as F
SHA2_BITS = 256

In [0]:
dbutils.widgets.text("catalog", "abcgroup", "Catalog")
dbutils.widgets.text("table_name", "date", "Table")

In [0]:
catalog = dbutils.widgets.get("catalog")
table_name = dbutils.widgets.get("table_name")

In [0]:
%run ./utilities

In [0]:
start_date = "2010-12-01"
end_date   = "2014-01-31"

In [0]:
df = (
    spark.sql(f"""
        SELECT explode(
            sequence(
                to_date('{start_date}'),
                to_date('{end_date}'),
                interval 1 day
            )
        ) AS full_date
    """)
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("month_short_name", F.date_format("full_date", "MMM"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter("full_date")))
    .withColumn("year_quarter", F.concat(F.col("year"), F.lit("-Q"), F.quarter("full_date")))
    .withColumn("date_sk", F.sha2(F.concat_ws("||", F.col("date_key"), F.col("full_date")), SHA2_BITS))
)

In [0]:
display(df.limit(5))

In [0]:
df.columns

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{gold_schema}.dim_{table_name}")